# TP2 - Binary Independence Retrieval & Latent Semantic Indexing  (09/03/2025)

In this TP, we introduce a probabilistic retrieval model named Binary Independence Retrieval (BIR) and a geometric model named Latent Semantic Indexing (LSI). We will implement the BIR and LSI models to retrieve the most relevant documents for input queries (terms or phrases) using the NASA document collection, as in TP1. Note that in the literature BIR is often mentioned as Binary Independence Model (BIM).

### Binary Independence Retrieval

BIR ranks the documents based on their probability of relevance with respect to a query. It defines a binary random variable $R$ for relevance, which is equal to 1 if the document is relevant for the query and 0 otherwise, and then estimates the probability
$p(R=1|d_{j},q)$ for each document $d_{j}$ and a given query $q$. To avoid arbitrary thresholds for this probability, it relies on the (log) odds ratio. This ratio is developed based on the Bayes theorem (see slides for analytical derivation). In practice, we will consider $R$ as a set of documents, initially assumed to be relevant. Its complementary set is given by $\bar{R}$ and contains all non-relevant documents in the collection. Consequently, $P(R|d_{j},q)$ is the probability that the document $d_{j}$ is relevant for the query $q$ and $P(\bar{R}|d_{j},q)$ is the probability that the document $d_{j}$ is non-relevant for the query $q$. The similarity measure $sim(d_{j},q)$ between the document $d_{j}$ and the query $q$ is defined as the ratio
\begin{equation}
sim(d_{j},q)= O(R|d_{j},q)= \frac{P(R|d_{j},q)}{P(\bar{R}|d_{j},q)}.
\end{equation}

By applying Bayes' rule, we get the following expression
\begin{equation}
sim(d_{j},q)=\frac{P(d_{j}|R,q)\cdot P(R|q)}{P(d_{j}|\bar{R},q)\cdot P(\bar{R|q})},
\end{equation}

where $P(d_{j}|R,q)$ corresponds to the probability of randomly selecting the document $d_{j}$ from the relevant set $R$ and $P(R|q)$ corresponds to the prior probability that a selected document is relevant. Likewise, $P(d_{j}|\bar{R})$ and $P(\bar{R}|q)$ are the analogous probabilities for the complementary set of non-relevant documents.

BIR also assumes term occurrences are binary and independent in the document. Hence, we will replace $d_{j}$ by its binary incidence term vector to proceed with the computation of the above quantities (see slides for analytical expression). By taking logarithms, we write the retrieval status value as 

\begin{equation}
sim(d_{j},q)=\sum_{i=1}^{k} w_{iq} \cdot w_{ij} \cdot \left( \log \frac{P(t_{i}|R,q)}{1-P(t_{i}|R,q)} +\log \frac{1-P(t_{i}|\bar{R},q)}{P(t_{i}|\bar{R},q)} \right),
\end{equation}
    
where  $P(t_{i}|R,q)$ corresponds to the probability that the index term $t_{i}$ is present in a document, randomly selected from the set $R$. $P(t_{i}|\bar{R},q)$ is the probability that term $t_{i}$ appears in a non-relevant document. 

We do not know the set $R$ in the beginning, hence we present an approach to estimate the previous probabilities. We initially assume that $P(t_{i}|R,q)$ is constant for all index terms, e.g., equal to $0.5$. We will approximate the distribution of index terms among non-relevant documents by the distribution of index terms across all documents in the collection. These two assumptions lead to the following:
 
\begin{equation}
P(t_{i}|R,q)=0.5, P(t_{i}|\bar{R},q)=\frac{n_{i}}{N},
\end{equation}
    
where $N$ is the total number of documents, and $n_{i}$ is the number of documents in which the
term $t_{i}$ appears. Based on this, we can now estimate the similarity score and retrieve an initial probabilistic ranking that is iteratively improved as it follows. 
    
Let $V$ be a subset of the documents initially retrieved and ranked by the probabilistic model, for instance, the top $r$ ranked documents where $r$ is a certain threshold. Moreover, let $V_{i}$ be the subset of $V$ composed by documents that contain the index term $t_{i}$. 
    
We iteratively improve the initially guessed values of $P(t_{i}|R,q)$ and $P(t_{i}|\bar{R},q)$. This is achieved by satisfying the following assumptions: we approximate $P(t_{i}|R,q)$ by the distribution of index term $t_{i}$ among the documents retrieved so far, and we approximate $P(t_{i}|\bar{R},q)$ by considering that all the non-retrieved documents are not relevant. Based on these assumptions, we express the updating rules as follows:
    
\begin{equation}
P(t_{i}|R,q):=\frac{|V_{i}|}{|V|}, P(t_{i}|\bar{R},q):=\frac{n_{i}-|V_{i}|}{N-|V|}.
\end{equation}
    
To avoid zero $P(t_{i}|R,q)$ and $P(t_{i}|\bar{R},q)$ for small values of $V$ and $V_{i}$, we add an adjustment factor and obtain:
    
\begin{equation}
P(t_{i}|R,q):=\frac{|V_{i}|+0.5}{|V|+1}, P(t_{i}|\bar{R},q):=\frac{n_{i}-|V_{i|}+0.5}{N-|V|+1}.
\end{equation}
    
An alternative is to take the fraction $\frac{n_{i}}{N}$ as the adjustment factor and get:

\begin{equation}
P(t_{i}|R,q):=\frac{|V_{i}|+\frac{n_{i}}{N}}{|V|+1},  P(t_{i}|\bar{R},q):=\frac{n_{i}-|V_{i}|+\frac{n_{i}}{N}}{N-|V|+1},
\end{equation}
    
where $|\cdot|$ means a number of set elements. 

### Latent semantic indexing model

The LSI model matches documents to a given query based on conceptual similarity instead of exact term matching. In this way, it retrieves relevant documents even when they are not explicitly indexed by query terms. For example, the IR model might return a document because it shares concepts with another document, which is relevant to the given query. LSI maps each document and query vector into a lower dimensional space, which is associated with concepts. The reduced space might be better for capturing conceptual similarities (e.g., synonyms) compared to the original one.
    
Let $k$ be the number of index terms in the collection of documents and $N$ be the the total number of documents. Define $M$ as a matrix with $k$ rows and $N$ columns. Each element $M_{ij}$ of this matrix is assigned a weight $w_{ij}$ associated with the index term $t_{i}$ and the document $d_{j}$. The weight $w_{ij}$ can be computed using the tf-idf weighting scheme.
 
LSI decomposes the term-document matrix $M$ using the singular value decomposition as it follows:
    
\begin{equation}
M=S \cdot  \Delta \cdot D^{T}.
\end{equation}
    
Matrix $S$ is the matrix of eigenvectors obtained from $M\cdot M^{T}$. Matrix $D$ is the matrix of eigenvectors derived from $ M^{T} \cdot M$. Matrix $ \Delta$ is an $r \times r$ diagonal matrix of singular values where $r=min(k,N)$ is the rank of $M$.
    
Consider now only the $l$ largest singular values of $ \Delta$ and keep them along with their corresponding columns in $S$ and $D$, respectively. The result is matrix $M_{l}$ given by
    
\begin{equation}
M_{l}=S_{l} \cdot  \Delta_{l} \cdot D_{l}^{T},
\end{equation}
    
where $l$, $l<r$, is the dimensionality of the reduced concept space. A good value for $l$ balances two opposing effects. One one hand, $l$ should be large enough to capture all the structure present in the real data. On the other hand, it should be small enough to filter out non-relevant details of data.
    
The relation between any two documents in the reduced space of dimensionality $l$ can be derived from the matrix $M_{l}^{T} \cdot M_{l}$, where the element $(i,j)$ quantifies the relationship between documents $d_{i}$ and $d_{j}$.
    
To rank the documents with regards to a given user query, we consider the query as a document in the original matrix $M$, and map it into the same reduced-dimensional space as the documents. 

### Recommended readings

BIR: https://nlp.stanford.edu/IR-book/pdf/11prob.pdf

LSI: https://nlp.stanford.edu/IR-book/html/htmledition/latent-semantic-indexing-1.html

### Exercises 

1. Use 15 articles from the NASA corpus to obtain raw data and apply the pre-processing steps as in TP1.
2. Compute the Boolean term document matrix, as well as its tf-idf version. 
3. Implement the BIR and LSI models based on the top $p$ stems. Provide different queries to each IR system. Compare the rankings of the relevant articles for the two models.
4. Compare the BIR and LSI results with the results from the boolean and vector models in TP1.

Implementation of required methods:

Create a class `ProbModelIR` that is initialized with parameters (root_docs_path, num_docs=15, p=50) and implements the methods:

1.  `termDocumentMatrixProbModel(self)`: reads all .txt from a specifed directory (where your corpus is), creates a boolean representation for each document, and constructs the term-document matrix.
2. `queryBooleanRepresentationProbModel(queries)`: returns the boolean representation of the query for BIR.
3. `rankingProbModel(self)` computes the similarity between a given query and the documents, and returns similarity values and filenames of the top  𝑁 relevant documents.

Create a class `LSIModelIR` that is initialized with parameters (root_docs_path, num_docs=15, p=50) and implements the methods:

1. `termDocumentMatrixLatentSemanticIndexing(self)`: reads all .txt from a specifed directory (where your corpus is), creates a latent semantic indexing representation for all documents and queries, and constructs the term-document matrix.
2. `queryVectorRepresentationLSIModel(queries)`: returns the vector representation of the query for the latent semantic indexing model.
3. `rankingLatentSemanticIndexing(self)`: computes the similarity between a given query and the documents, and returns similarity values and filenames of the top $N$ relevant documents. Note that a query is treated as a short document in term-document matrix (see slide  IR.02, p.49).